# PneuFed — Colab Backend (ResNet18 Round-20)
Run each cell **top to bottom**.
- Uses the real loaded ResNet18 model weights from Round-20 federated training.
- Never uses mock or placeholder predictions.
- Auto-cleans port 5000 before starting to prevent port conflicts.

In [ ]:
# CELL 1: Free Port 5000 + Install Dependencies
import subprocess
subprocess.run(['fuser', '-k', '5000/tcp'], capture_output=True)
print('✅ Port 5000 cleared')

!pip install -q flask flask-cors pyngrok torch torchvision pillow
print('✅ Dependencies installed')

In [ ]:
# CELL 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

In [ ]:
# CELL 3: Verify and Load Real ResNet18 Round-20 Model
import os
import torch
import torch.nn as nn
from torchvision import models

# Check standard Drive locations
candidates = [
    '/content/drive/MyDrive/PneuFed/local_models/global_model_round_20.pth',
    '/content/drive/MyDrive/PneuFed/models/global_model_round_20.pth'
]

MODEL_PATH = next((p for p in candidates if os.path.exists(p)), None)

# Fallback search in MyDrive if moved
if not MODEL_PATH:
    print('Searching for global_model_round_20.pth in Google Drive...')
    for root, dirs, files in os.walk('/content/drive/MyDrive/PneuFed'):
        if 'global_model_round_20.pth' in files:
            MODEL_PATH = os.path.join(root, 'global_model_round_20.pth')
            break

if not MODEL_PATH or not os.path.exists(MODEL_PATH):
    raise FileNotFoundError('❌ Checkpoint global_model_round_20.pth not found in Google Drive. Please verify the folder structure in Drive.')

print(f'✅ Found checkpoint: {MODEL_PATH}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Instantiate ResNet18 with 2 output classes (NORMAL and PNEUMONIA)
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)

checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint)
model = model.to(device)
model.eval()

print('✅ Real ResNet18 Round-20 model loaded and ready in eval mode!')

In [ ]:
# CELL 4: Image Preprocessing (224x224, ImageNet Normalization)
import io
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image

CLASS_NAMES = ['NORMAL', 'PNEUMONIA']

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def predict_image(image_bytes):
    image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    tensor = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(tensor)
        probs = F.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probs, dim=1)
    label = CLASS_NAMES[predicted_idx.item()]
    conf_pct = round(confidence.item() * 100, 2)
    return label, conf_pct

print('✅ Preprocessing and inference function defined')

In [ ]:
# CELL 5: Flask API with CORS and Endpoints
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app, resources={r'/*': {'origins': '*'}})

@app.after_request
def add_cors_headers(resp):
    resp.headers['Access-Control-Allow-Origin'] = '*'
    resp.headers['Access-Control-Allow-Headers'] = 'Content-Type, ngrok-skip-browser-warning, Authorization'
    resp.headers['Access-Control-Allow-Methods'] = 'GET, POST, OPTIONS'
    return resp

@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status': 'ok',
        'model': 'global_model_round_20',
        'model_loaded': True
    }), 200

@app.route('/predict', methods=['POST'])
def predict():
    if 'image' not in request.files:
        return jsonify({'error': 'No image field in request'}), 400
    file = request.files['image']
    if not file or file.filename == '':
        return jsonify({'error': 'Empty filename'}), 400
    try:
        image_bytes = file.read()
        prediction, confidence = predict_image(image_bytes)
        return jsonify({
            'prediction': prediction,
            'confidence': confidence,
            'model': 'Global Round-20 ResNet18'
        }), 200
    except Exception as e:
        return jsonify({'error': f'Prediction failed: {str(e)}'}), 500

print('✅ Flask routes defined: GET /health, POST /predict')

In [ ]:
# CELL 6: Start Flask in Background Thread (Safely)
import threading, time, subprocess

subprocess.run(['fuser', '-k', '5000/tcp'], capture_output=True)
time.sleep(1)

def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(2)
print('✅ Flask server is running in background on port 5000')

In [ ]:
# CELL 7: Authenticate ngrok (Token input is hidden)
from getpass import getpass
from pyngrok import ngrok

ngrok.kill()
token = getpass('🔐 Enter your ngrok authtoken (hidden): ')
ngrok.set_auth_token(token)
print('✅ ngrok authenticated successfully')

In [ ]:
# CELL 8: Connect ngrok Tunnel & Self-Test
from pyngrok import ngrok
import requests

tunnel = ngrok.connect(5000)
PUBLIC_API_URL = tunnel.public_url

print('=' * 60)
print('🚀 BACKEND IS LIVE!')
print(f'Public API URL : {PUBLIC_API_URL}')
print(f'Health Check   : {PUBLIC_API_URL}/health')
print(f'Predict Route  : {PUBLIC_API_URL}/predict')
print('=' * 60)

# Automated self-test check
try:
    resp = requests.get(f'{PUBLIC_API_URL}/health', headers={'ngrok-skip-browser-warning': 'true'}, timeout=10)
    if resp.status_code == 200:
        print('✅ Health check PASSED:', resp.json())
        print('\n📋 NEXT STEP: Copy the Public API URL above.')
        print('Open https://pbl-umber.vercel.app, click "Update URL", and paste it.')
    else:
        print('❌ Health check returned:', resp.status_code, resp.text)
except Exception as e:
    print('❌ Health check failed:', e)